In [1]:

import sympy as sp
import numpy as np
from scipy.stats import ortho_group
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.linalg import expm, logm

def skew_np(vec_):
    vec = np.squeeze(vec_.reshape(3,))
    """Returns the skew-symmetric matrix [vec]_x for a 3x1 vector."""
    return np.array([
        [0, -vec[2], vec[1]],
        [vec[2], 0, -vec[0]],
        [-vec[1], vec[0], 0]
    ])

# Generate a random 3x3 orthogonal matrix
# random_orthogonal_matrix = ortho_group.rvs(3)
# random_orthogonal_matrix_2 = ortho_group.rvs(3)
v_init = np.array([1, 2, 3]).reshape(3, 1)
p_init = np.array([4, 5, 6]).reshape(3, 1)
# print(random_orthogonal_matrix)
random_orthogonal_matrix_2 = np.array([[-0.69835356 , 0.58469477 , 0.41283692],
                                        [-0.71568049, -0.57863135, -0.39113579],
                                        [ 0.01018534 ,-0.5686104  , 0.8225439 ]])
random_orthogonal_matrix = np.array([[ 0.04947645 , 0.24943273, -0.96712739],
                                        [ 0.16374184 ,-0.9572376 , -0.23850531],
                                        [-0.98526174, -0.14655883, -0.08820329]])

np.set_printoptions(precision=4)

In [2]:


def h_acc(state, a_n, omega_n, i):
    R_imu, v_imu, p_imu, b_omega, b_a, R_c, p_c = state[:3], state[3:6].reshape(3, 1), \
        state[6:9].reshape(3, 1), state[9:12], state[12:15], state[15:18], state[18:21]
    
    R_c = expm(skew_np(state[15:18].reshape(3,))) @ random_orthogonal_matrix
    print("the term 4/7 prior \n", (R_c.T @ skew_np(p_c.reshape(3,))))

    previous_chi = np.vstack((
                            np.hstack((random_orthogonal_matrix_2, v_init, p_init)),
                            np.hstack((np.zeros((2, 3)), np.eye(2)))
                        ))
    
    skew_5_5 = np.vstack((
        np.hstack((skew_np(state[0:3].reshape(3,)), v_imu, p_imu)),
        np.zeros((2, 5))
    ))
    
    R_imu = (expm(skew_5_5) @ previous_chi )[:3, :3]

    b_vector = (( R_imu.T @ v_imu).flatten() + np.cross(omega_n - b_omega, p_c)).reshape(3,)
    
    v_c = R_c.T @ (( R_imu.T @ v_imu).flatten() + np.cross(omega_n - b_omega, p_c))

    if(i == 0):
        print("the term 1/7 is \n", np.zeros((2, 3)))
        print("the term 2/7 is \n", (R_c.T  @ R_imu.T).T)
        print("the term 3/7 is \n", np.zeros((2, 3)))
        print("the term 4/7 is \n", (R_c.T @ skew_np(p_c.reshape(3,))).T)
        print("the term 5/7 is \n", np.zeros((2, 3)))
        print("the term 6/7 B \n", (R_c.T @ skew_np (b_vector)).T)
        print("FOR term 7/7  \n", (R_c.T @ skew_np(omega_n - b_omega)).T)

    obs_paper = v_c
    return obs_paper

def compute_jacobian_paper(x_0, a_n, omega_n, d=1e-9):
    # Define small perturbations
    n = len(x_0)
    jacobian = np.zeros((3, n))

    # Compute the observation at the nominal state
    obs_nominal = h_acc(x_0, a_n, omega_n,10)

    for i in range(n):
        # Compute the observation with perturbed state
        x_0_perturbed = np.copy(x_0)
        x_0_perturbed[i] += d
        obs_perturbed = h_acc(x_0_perturbed, a_n, omega_n, i)
        # Calculate Jacobian element by finite difference
        jacobian[:, i] = (obs_perturbed - obs_nominal) / d

    return jacobian

def h_acc_new(state, a_n, omega_n, i):
    R_imu1, v_imu, p_imu, b_omega, b_a, R_c_old, p_c = state[:3], state[3:6].reshape(3, 1), \
        state[6:9].reshape(3, 1), state[9:12], state[12:15], state[15:18], state[18:21]
    
    R_c = expm(skew_np(state[15:18].reshape(3,))) @ random_orthogonal_matrix
    previous_chi = np.vstack((
                            np.hstack((random_orthogonal_matrix_2, v_init, p_init)),
                            np.hstack((np.zeros((2, 3)), np.eye(2)))
                        ))
    
    skew_5_5 = np.vstack((
        np.hstack((skew_np(state[0:3].reshape(3,)), v_imu, p_imu)),
        np.zeros((2, 5))
    ))
    
    R_imu = (expm(skew_5_5) @ previous_chi )[:3, :3]
    vee_0 = (omega_n - b_omega).reshape(3,)
    omega_p = np.cross(vee_0, p_c)

    alpha1 = (a_n - b_a + np.cross(vee_0, omega_p)).reshape(3,)
    alpha2 = (R_imu.T @ v_imu ).flatten() + np.cross(omega_n - b_omega, p_c)
    alpha3 = (vee_0).reshape(3,1)


    v_c = R_c.T @ alpha2

    omega_car = R_c.T @ alpha3
    a_car = R_c.T @ alpha1

    ############## find analytical solution #################
    #  only print one time
    if(i == 0):
        ############ for check the derivation  ################
        M1 = np.array([0, 1, 0]).reshape(1, 3)
        M2 = np.array([1, 0, 0]).reshape(1, 3)
        M3 = np.array([0, 0, 1]).reshape(1, 3)
        print("the term 1/7 is \n", np.zeros((1, 3)))

        # for term 2/7 correct
        print("the 2/7 term : \n", - M2 @ (R_c.T @ R_imu.T) * (omega_car[2]))
        print("the term 3/7 is \n", np.zeros((1, 3)))

        vee = (omega_n - b_omega).reshape(3,)
        pee = p_c.reshape(3,)

        term_4_7 =  M1 @ R_c.T @ ( - np.outer(vee, pee) - np.squeeze((omega_n - b_omega).dot(p_c)) * np.eye(3) + (2 * np.outer(pee, vee))) \
                    + (1 * M2 @ R_c.T @ skew_np(p_c)) * (-omega_car[2]) \
                    + (-1 * M3 @ R_c.T) * (-v_c[0])
        
        print("the 4/7 term : \n", term_4_7)

        #  for term 5/7 correct
        term_5_7 = -1 * M1 @ R_c.T 
        print("the 5/7 term : \n", term_5_7)

        #  for term 6/7  correct
        term_6_7 = M1 @ R_c.T @ skew_np(alpha1) \
                   - np.squeeze(np.array([1,0,0]) @ R_c.T @ skew_np(alpha2)) * omega_car[2] \
                   - v_c[0] * (M3 @ R_c.T @ skew_np(alpha3))
        # print("pre 67", np.squeeze(np.array([1,0,0]) @ R_c.T @ skew_np(alpha2)))
        print("the 6/7 term : \n", term_6_7)

        #  for term 7/7 correct
        term_7_7 = M1 @ R_c.T @ skew_np(omega_n - b_omega) @ skew_np(omega_n - b_omega) \
                    + M2 @ R_c.T @ skew_np( - omega_n + b_omega) * (omega_car[2])
        print("the 7/7 term : \n", term_7_7)

        # https://arxiv.org/pdf/1312.0788 equation (12)
        omega_real = omega_n - b_omega
        term_7_7_simplified = M1 @ R_c.T @ (np.outer(omega_real,omega_real) - (omega_real.T).dot(omega_real) * np.eye(3)) \
                            + M2 @ R_c.T @ skew_np( - omega_real) * (omega_car[2])
        print("the 7/7 term simplified : \n", term_7_7_simplified)

    obs_new = a_car[1] - v_c[0] * omega_car[2]
    return  obs_new

def compute_jacobian_new_obser(x_0, a_n, omega_n, d=1e-9):
    # Define small perturbations
    n = len(x_0)
    jacobian = np.zeros((n))

    # Compute the observation at the nominal state
    obs_nominal = h_acc_new(x_0, a_n, omega_n, 2)

    for i in range(n):
        # Compute the observation with perturbed state
        x_0_perturbed = np.copy(x_0)
        x_0_perturbed[i] += d
        obs_perturbed = h_acc_new(x_0_perturbed, a_n, omega_n, i)
        # Calculate Jacobian element by finite difference
        # print(obs_perturbed)
        jacobian[i] = (obs_perturbed - obs_nominal) / d

    return jacobian

# Example usage:
chi_imu = np.array([0.0, 0.0, 0.0])
v_imu = np.array([0.0, 0.0, 0.0])
p_imu = np.array([0.0, 0.0, 0.0])
b_a = np.array([0.0, 0.0, 0.0])
b_omega = np.array([0.5, 0.01, 0.01])
chi_c = np.array([0.0, 0.0, 0.0])
p_c = np.array([0.1, 0.1, 0.1])

# Nominal state

a_n = np.array([1.0, 1.0, 1.0])
omega_n = np.array([0.5, 0.1, 1.0])
x_0 = np.concatenate((chi_imu, v_imu, p_imu, b_omega, b_a, chi_c, p_c))

In [3]:
compute_jacobian_paper(x_0, a_n, omega_n).T
# compute_jacobian_new_obser(x_0, a_n, omega_n).reshape(21, 1)

the term 4/7 prior 
 [[ 0.1149 -0.1035 -0.0114]
 [-0.0811 -0.0396  0.1207]
 [-0.015   0.0879 -0.0729]]
the term 4/7 prior 
 [[ 0.1149 -0.1035 -0.0114]
 [-0.0811 -0.0396  0.1207]
 [-0.015   0.0879 -0.0729]]
the term 1/7 is 
 [[0. 0. 0.]
 [0. 0. 0.]]
the term 2/7 is 
 [[-0.3456 -0.7944  0.4995]
 [ 0.2552  0.4327  0.8647]
 [-0.903   0.4263  0.0532]]
the term 3/7 is 
 [[0. 0. 0.]
 [0. 0. 0.]]
the term 4/7 is 
 [[ 0.1149 -0.0811 -0.015 ]
 [-0.1035 -0.0396  0.0879]
 [-0.0114  0.1207 -0.0729]]
the term 5/7 is 
 [[0. 0. 0.]
 [0. 0. 0.]]
the term 6/7 B 
 [[ 0.0961  0.0231  0.0109]
 [ 0.0891  0.0154 -0.0008]
 [ 0.0196 -0.0615 -0.1172]]
FOR term 7/7  
 [[ 0.2508 -0.9345 -0.2282]
 [-0.049  -0.2469  0.9575]
 [ 0.0045  0.0224 -0.087 ]]
the term 4/7 prior 
 [[ 0.1149 -0.1035 -0.0114]
 [-0.0811 -0.0396  0.1207]
 [-0.015   0.0879 -0.0729]]
the term 4/7 prior 
 [[ 0.1149 -0.1035 -0.0114]
 [-0.0811 -0.0396  0.1207]
 [-0.015   0.0879 -0.0729]]
the term 4/7 prior 
 [[ 0.1149 -0.1035 -0.0114]
 [-0.0811 -0.0

array([[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [-3.4557e-01, -7.9439e-01,  4.9953e-01],
       [ 2.5522e-01,  4.3270e-01,  8.6466e-01],
       [-9.0302e-01,  4.2628e-01,  5.3215e-02],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 1.1490e-01, -8.1068e-02, -1.5030e-02],
       [-1.0347e-01, -3.9599e-02,  8.7892e-02],
       [-1.1427e-02,  1.2067e-01, -7.2862e-02],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
       [ 9.6067e-02,  2.3124e-02,  1.0879e-02],
       [ 8.9119e-02,  1.5435e-02, -7.6585e-04],
       [ 1.9635e-02, -6.1458e-02, -1.1721e-01],
       [ 2.5078e-01, -9.3447e-01, -2.2818e-01],
       [-4.8982e-02, -2.4694e-01,  9.5746e-01],
       [ 4.4529e-03,  2.2449e-02, -8.704

In [4]:
import numpy as np

alpha2_phuc = np.array([0.5, 1.0, 3.0])
epsilon_r0 = np.array([0.0, 0.0, 0.0])
def fx(epsilon_r):
    R_c = expm(skew_np(epsilon_r.reshape(3,))) @ random_orthogonal_matrix
    # alpha2_phuc = (R_imu.T @ v_imu).flatten() + np.cross(omega_n - b_omega, p_c)
    v_c = R_c.T @ alpha2_phuc
    jacobian_all = R_c.T @ skew_np(alpha2_phuc)
    print("analytical: ", np.array([1, 0, 0]).reshape(1, 3) @ jacobian_all)
    return v_c[0]


fx_0 = fx(epsilon_r0)
#  find jacobian w.r.t bias omega
jacobian = np.zeros(3)
for i in range(3):
    d = 1e-9
    # Compute the observation with perturbed state
    epsilon_r_cp = np.copy(epsilon_r0)
    epsilon_r_cp[i] +=  d
    fx_per = fx(epsilon_r_cp)
    jacobian[i] = (fx_per - fx_0) / d

print("numerical: ", jacobian)


analytical:  [[ 1.4765 -0.6411 -0.0324]]
analytical:  [[ 1.4765 -0.6411 -0.0324]]
analytical:  [[ 1.4765 -0.6411 -0.0324]]
analytical:  [[ 1.4765 -0.6411 -0.0324]]
numerical:  [ 1.4765 -0.6411 -0.0324]


In [5]:
import numpy as np

def compute_jacobian(state, a_n, omega_n):
    # Define M1, M2, M3 as row vectors
    M1 = np.array([0, 1, 0]).reshape(1, 3)
    M2 = np.array([1, 0, 0]).reshape(1, 3)
    M3 = np.array([0, 0, 1]).reshape(1, 3)
    
    # Extract relevant state components
    R_c = state[15:18].reshape(3, 3)  # Assuming R_c is stored directly as a matrix
    p_c = state[18:21].reshape(3, 1)
    b_omega = state[9:12].reshape(3, 1)
    omega_real = omega_n.reshape(3, 1) - b_omega
    omega_car = R_c.T @ omega_real
    
    # compute terms for old observation
    term_1_7_old = np.zeros((2, 3))
    term_2_7_old = (R_c.T @ R_imu.T).T
    term_3_7_old = np.zeros((2, 3))
    term_4_7_old = (R_c.T @ skew_np(p_c.reshape(3,))).T
    term_5_7_old = np.zeros((2, 3))
    term_6_7_old = (R_c.T @ skew_np(b_vector)).T
    term_7_7_old = (R_c.T @ skew_np(omega_n - b_omega)).T

    J_old = np.hstack([term_1_7_old, term_2_7_old, term_3_7_old, term_4_7_old, term_5_7_old, term_6_7_old, term_7_7_old])


    # Compute terms for new observation
    term_1_7_new = np.zeros((1, 3))
    term_2_7_new = - M2.T @ (R_c.T @ R_c.T) * omega_car[2]
    term_3_7_new = np.zeros((1, 3))
    
    term_4_7_new = M1.T @ R_c.T @ (-np.outer(omega_real.flatten(), p_c.flatten()) - np.dot(omega_real.T, p_c) * np.eye(3) + 2 * np.outer(p_c.flatten(), omega_real.flatten())) \
               + M2.T @ R_c.T @ skew_np(p_c.flatten()) * (-omega_car[2]) \
               + (-M3.T @ R_c.T) * (-omega_car[0])
    
    term_5_7_new = -M1.T @ R_c.T
    
    term_6_7_new = M1.T @ R_c.T @ skew_np(a_n - b_omega + np.cross(omega_real.flatten(), np.cross(omega_real.flatten(), p_c.flatten()))) \
               - np.squeeze(M2 @ R_c.T @ skew_np(omega_real.flatten())) * omega_car[2] \
               - omega_car[0] * (M3 @ R_c.T @ skew_np(omega_real.flatten()))
    
    term_7_7_new = M1.T @ R_c.T @ (np.outer(omega_real.flatten(), omega_real.flatten()) - np.dot(omega_real.T, omega_real) * np.eye(3)) \
               + M2.T @ R_c.T @ skew_np(-omega_real.flatten()) * omega_car[2]
    
    # Concatenate into Jacobian matrix (3x21)
    J_new = np.hstack([term_1_7_new, term_2_7_new, term_3_7_new, term_4_7_new, term_5_7_new, term_6_7_new, term_7_7_new])
    
    return J

def skew_np(v):
    """ Returns the skew-symmetric matrix of a 3D vector. """
    return np.array([[0, -v[2], v[1]],
                     [v[2], 0, -v[0]],
                     [-v[1], v[0], 0]])
